# Regularization Techniques in Deep Learning: Dropout, L1/L2

This notebook explores various regularization techniques used in deep learning to prevent overfitting and improve model generalization. We will focus specifically on:

- L1 Regularization (Lasso)
- L2 Regularization (Ridge)
- Elastic Net (combined L1+L2)
- Dropout Regularization

Each technique will be explained conceptually, demonstrated with code, and compared through experiments.

## 1. Import Required Libraries

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.datasets import make_classification, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

# For deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input, Conv2D, MaxPooling2D, Flatten

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Understanding Regularization in Deep Learning

Regularization is a set of techniques used to prevent overfitting in machine learning models. Overfitting occurs when a model learns the training data too well, capturing noise and specific patterns that don't generalize to new data.

### Why is Regularization Needed?

- **Without regularization**: Models can become too complex and fit the training data perfectly but perform poorly on unseen data
- **With regularization**: Models are constrained to be simpler, leading to better generalization

Let's first create a simple dataset and visualize the concept of overfitting versus appropriate regularization.

In [ ]:
# Create a synthetic regression dataset with some noise
def generate_nonlinear_data(n_samples=500):
    np.random.seed(42)
    X = np.linspace(-3, 3, n_samples).reshape(-1, 1)
    y = 0.5 * X**2 + X + 2 + np.random.normal(0, 1, size=X.shape)
    return X, y

X, y = generate_nonlinear_data(200)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.7, label='Training Data')
plt.scatter(X_test, y_test, alpha=0.7, label='Test Data')
plt.title('Synthetic Regression Dataset')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
# Function to create models with different levels of complexity and regularization
def create_models():
    # Underfit model (too simple)
    underfit_model = Sequential([
        Dense(1, activation='linear', input_shape=(1,))
    ])
    
    # Just-right model (balanced complexity)
    good_model = Sequential([
        Dense(16, activation='relu', input_shape=(1,)),
        Dense(8, activation='relu'),
        Dense(1, activation='linear')
    ])
    
    # Overfit model (too complex, no regularization)
    overfit_model = Sequential([
        Dense(128, activation='relu', input_shape=(1,)),
        Dense(128, activation='relu'),
        Dense(128, activation='relu'),
        Dense(1, activation='linear')
    ])
    
    # Regularized model (complex but with regularization)
    regularized_model = Sequential([
        Dense(128, activation='relu', input_shape=(1,), 
              kernel_regularizer=regularizers.l2(0.01)),
        Dense(128, activation='relu', 
              kernel_regularizer=regularizers.l2(0.01)),
        Dense(128, activation='relu', 
              kernel_regularizer=regularizers.l2(0.01)),
        Dense(1, activation='linear')
    ])
    
    # Compile all models
    models = {
        'Underfit (Too Simple)': underfit_model,
        'Good Fit (Balanced)': good_model,
        'Overfit (Too Complex)': overfit_model,
        'Regularized (Complex with L2)': regularized_model
    }
    
    for model in models.values():
        model.compile(optimizer='adam', loss='mse')
        
    return models

# Train and evaluate different models
models = create_models()

# Dictionary to store training histories
histories = {}

# Train each model
for name, model in models.items():
    print(f"Training {name}...")
    history = model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=0
    )
    histories[name] = history

In [ ]:
# Visualize predictions for each model
plt.figure(figsize=(14, 10))

# Generate a range of X values for smoother curve visualization
X_range = np.linspace(-4, 4, 1000).reshape(-1, 1)

for i, (name, model) in enumerate(models.items()):
    plt.subplot(2, 2, i+1)
    
    # Plot training and test data
    plt.scatter(X_train, y_train, alpha=0.4, label='Training Data')
    plt.scatter(X_test, y_test, alpha=0.4, label='Test Data')
    
    # Plot model predictions
    y_pred = model.predict(X_range, verbose=0)
    plt.plot(X_range, y_pred, color='red', linewidth=2, label='Prediction')
    
    plt.title(f'{name}\nTest MSE: {mean_squared_error(y_test, model.predict(X_test, verbose=0)):.4f}')
    plt.xlabel('X')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize training and validation loss
plt.figure(figsize=(14, 6))

for name, history in histories.items():
    plt.plot(history.history['loss'], label=f'{name} - Training')
    plt.plot(history.history['val_loss'], '--', label=f'{name} - Validation')

plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

### Key Takeaways on Regularization

From the visualizations above, we can observe:

1. **Underfit model**: Too simple, cannot capture the underlying pattern in the data
2. **Good fit model**: Balanced complexity, captures the pattern without overfitting
3. **Overfit model**: Too complex, fits the training data almost perfectly but generalizes poorly to test data
4. **Regularized model**: Uses L2 regularization to constrain weights, achieving better generalization despite high model complexity

This demonstrates the fundamental need for regularization: it allows us to use complex models while preventing them from memorizing the training data.

Next, we'll explore specific regularization techniques in detail.

## 3. L1 Regularization (Lasso)

L1 regularization, also known as Lasso regression (Least Absolute Shrinkage and Selection Operator), adds the absolute value of weights as a penalty term to the loss function.

### Mathematical Formulation

For a model with parameters W, the L1 regularized loss function is:

$$\text{Loss}_{L1} = \text{Loss}_{\text{original}} + \lambda \sum_{i} |w_i|$$

Where:
- $\text{Loss}_{\text{original}}$ is the original loss function (e.g., MSE, cross-entropy)
- $\lambda$ is the regularization strength (hyperparameter)
- $\sum_{i} |w_i|$ is the sum of absolute values of all weights

### Key Properties of L1 Regularization:

1. **Feature Selection**: Tends to drive weights of unimportant features to exactly zero
2. **Sparse Models**: Results in sparse models where only important features have non-zero weights
3. **Robust to Outliers**: Less sensitive to outliers than L2 regularization

Let's implement and visualize models with L1 regularization.

In [ ]:
# Create a classification dataset with some irrelevant features
X, y = make_classification(
    n_samples=1000, 
    n_features=20,  # 20 features, but not all are important
    n_informative=5,  # Only 5 features are actually informative 
    n_redundant=5,
    n_repeated=0,
    n_classes=2,
    random_state=42
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create models with different L1 regularization strengths
def create_l1_models():
    models = {}
    l1_strengths = [0, 0.0001, 0.001, 0.01, 0.1]
    
    for strength in l1_strengths:
        name = f"L1 (λ={strength})"
        model = Sequential([
            Dense(64, activation='relu', input_shape=(20,),
                  kernel_regularizer=regularizers.l1(strength)),
            Dense(32, activation='relu',
                  kernel_regularizer=regularizers.l1(strength)),
            Dense(1, activation='sigmoid')
        ])
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        models[name] = model
    
    return models

# Train L1 regularized models
l1_models = create_l1_models()
l1_histories = {}

for name, model in l1_models.items():
    print(f"Training {name}...")
    history = model.fit(
        X_train_scaled, y_train,
        epochs=50,
        batch_size=32,
        validation_data=(X_test_scaled, y_test),
        verbose=0
    )
    l1_histories[name] = history

In [ ]:
# Extract and visualize weights from L1 models
plt.figure(figsize=(15, 10))

for i, (name, model) in enumerate(l1_models.items()):
    # Extract weights from the first layer
    weights = model.layers[0].get_weights()[0]
    
    # Plot weight distribution
    plt.subplot(2, 3, i+1)
    plt.hist(weights.flatten(), bins=50, alpha=0.7)
    plt.title(f"{name}\nWeight Distribution")
    plt.xlabel("Weight Value")
    plt.ylabel("Count")
    
    # Add text showing zero weights percentage
    zero_weights = np.sum(np.abs(weights) < 1e-10) / weights.size * 100
    plt.text(0.05, 0.95, f"Zero weights: {zero_weights:.2f}%",
             transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Visualize accuracy on test data for L1 models
val_accuracies = [l1_histories[name].history['val_accuracy'][-1] for name in l1_models.keys()]
names = list(l1_models.keys())

plt.figure(figsize=(10, 6))
plt.bar(names, val_accuracies, color='skyblue')
plt.title('Test Accuracy with Different L1 Regularization Strengths')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.ylim(0.7, 1.0)
plt.xticks(rotation=45)
plt.grid(axis='y')

for i, v in enumerate(val_accuracies):
    plt.text(i, v + 0.01, f"{v:.4f}", ha='center')

plt.tight_layout()
plt.show()

## 4. L2 Regularization (Ridge)

L2 regularization, also known as Ridge regression or weight decay, adds the sum of squared weights as a penalty term to the loss function.

### Mathematical Formulation

For a model with parameters W, the L2 regularized loss function is:

$$\text{Loss}_{L2} = \text{Loss}_{\text{original}} + \lambda \sum_{i} w_i^2$$

Where:
- $\text{Loss}_{\text{original}}$ is the original loss function
- $\lambda$ is the regularization strength (hyperparameter)
- $\sum_{i} w_i^2$ is the sum of squared weights

### Key Properties of L2 Regularization:

1. **Weight Shrinking**: Tends to shrink all weights toward zero, but rarely makes them exactly zero
2. **Handles Correlated Features**: Works well when input features are correlated
3. **Stability**: Generally more numerically stable than L1 regularization

Let's implement and visualize models with L2 regularization.

In [ ]:
# Create models with different L2 regularization strengths
def create_l2_models():
    models = {}
    l2_strengths = [0, 0.0001, 0.001, 0.01, 0.1]
    
    for strength in l2_strengths:
        name = f"L2 (λ={strength})"
        model = Sequential([
            Dense(64, activation='relu', input_shape=(20,),
                  kernel_regularizer=regularizers.l2(strength)),
            Dense(32, activation='relu',
                  kernel_regularizer=regularizers.l2(strength)),
            Dense(1, activation='sigmoid')
        ])
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        models[name] = model
    
    return models

# Train L2 regularized models
l2_models = create_l2_models()
l2_histories = {}

for name, model in l2_models.items():
    print(f"Training {name}...")
    history = model.fit(
        X_train_scaled, y_train,
        epochs=50,
        batch_size=32,
        validation_data=(X_test_scaled, y_test),
        verbose=0
    )
    l2_histories[name] = history

In [ ]:
# Extract and visualize weights from L2 models
plt.figure(figsize=(15, 10))

for i, (name, model) in enumerate(l2_models.items()):
    # Extract weights from the first layer
    weights = model.layers[0].get_weights()[0]
    
    # Plot weight distribution
    plt.subplot(2, 3, i+1)
    plt.hist(weights.flatten(), bins=50, alpha=0.7)
    plt.title(f"{name}\nWeight Distribution")
    plt.xlabel("Weight Value")
    plt.ylabel("Count")
    
    # Calculate standard deviation of weights
    std_dev = np.std(weights)
    plt.text(0.05, 0.95, f"Std Dev: {std_dev:.6f}",
             transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Visualize training and validation loss for L2 models
plt.figure(figsize=(12, 8))

for name, history in l2_histories.items():
    plt.plot(history.history['val_loss'], label=f"{name}")

plt.title('Validation Loss with Different L2 Regularization Strengths')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Visualize accuracy on test data for L2 models
val_accuracies = [l2_histories[name].history['val_accuracy'][-1] for name in l2_models.keys()]
names = list(l2_models.keys())

plt.figure(figsize=(10, 6))
plt.bar(names, val_accuracies, color='lightgreen')
plt.title('Test Accuracy with Different L2 Regularization Strengths')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.ylim(0.7, 1.0)
plt.xticks(rotation=45)
plt.grid(axis='y')

for i, v in enumerate(val_accuracies):
    plt.text(i, v + 0.01, f"{v:.4f}", ha='center')

plt.tight_layout()
plt.show()

### Comparing L1 and L2 Regularization

Let's directly compare the effect of L1 and L2 regularization on the weights of our model:

| **L1 Regularization** | **L2 Regularization** |
|----------------------|----------------------|
| Results in sparse weights (many exactly zero) | Weights are small but rarely exactly zero |
| Performs feature selection implicitly | All features are usually kept but with smaller weights |
| Better when many features are irrelevant | Better for handling correlated features |
| Penalty is the sum of absolute weight values | Penalty is the sum of squared weight values |

The choice between L1 and L2 should be guided by your specific needs:
- Choose L1 when you suspect many features are irrelevant and want automatic feature selection
- Choose L2 when you want to keep all features but reduce the impact of any single feature

## 5. Elastic Net (L1+L2 Combined)

Elastic Net combines L1 and L2 regularization to get the best of both worlds. It adds both the sum of absolute weights (L1 term) and the sum of squared weights (L2 term) to the loss function.

### Mathematical Formulation

$$\text{Loss}_{\text{ElasticNet}} = \text{Loss}_{\text{original}} + \lambda_1 \sum_{i} |w_i| + \lambda_2 \sum_{i} w_i^2$$

Where:
- $\lambda_1$ and $\lambda_2$ are the regularization strengths for L1 and L2 terms, respectively

### Key Properties of Elastic Net:

1. **Combined Benefits**: Gets both the feature selection properties of L1 and the stability of L2
2. **Handles Groups of Correlated Features**: Can select entire groups of correlated variables
3. **Flexibility**: Allows fine-tuning the balance between L1 and L2 effects

Let's implement Elastic Net regularization in our models.

In [ ]:
# Create models with Elastic Net regularization (L1 + L2)
def create_elastic_net_models():
    models = {}
    
    # Different combinations of L1 and L2 regularization
    combinations = [
        ("No Regularization", 0, 0),
        ("L1 Only (λ=0.01)", 0.01, 0),
        ("L2 Only (λ=0.01)", 0, 0.01),
        ("Elastic Net (L1=0.005, L2=0.005)", 0.005, 0.005),
        ("Elastic Net (L1=0.01, L2=0.01)", 0.01, 0.01)
    ]
    
    for name, l1_strength, l2_strength in combinations:
        # Create regularizer
        if l1_strength > 0 and l2_strength > 0:
            reg = regularizers.l1_l2(l1=l1_strength, l2=l2_strength)
        elif l1_strength > 0:
            reg = regularizers.l1(l1_strength)
        elif l2_strength > 0:
            reg = regularizers.l2(l2_strength)
        else:
            reg = None
            
        model = Sequential([
            Dense(64, activation='relu', input_shape=(20,),
                  kernel_regularizer=reg),
            Dense(32, activation='relu',
                  kernel_regularizer=reg),
            Dense(1, activation='sigmoid')
        ])
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        models[name] = model
    
    return models

# Train Elastic Net models
elastic_net_models = create_elastic_net_models()
elastic_net_histories = {}

for name, model in elastic_net_models.items():
    print(f"Training {name}...")
    history = model.fit(
        X_train_scaled, y_train,
        epochs=50,
        batch_size=32,
        validation_data=(X_test_scaled, y_test),
        verbose=0
    )
    elastic_net_histories[name] = history

In [ ]:
# Extract and visualize weights from Elastic Net models
plt.figure(figsize=(15, 12))

for i, (name, model) in enumerate(elastic_net_models.items()):
    # Extract weights from the first layer
    weights = model.layers[0].get_weights()[0]
    
    # Plot weight distribution
    plt.subplot(3, 2, i+1)
    plt.hist(weights.flatten(), bins=50, alpha=0.7)
    plt.title(f"{name}\nWeight Distribution")
    plt.xlabel("Weight Value")
    plt.ylabel("Count")
    
    # Calculate statistics
    std_dev = np.std(weights)
    zero_weights = np.sum(np.abs(weights) < 1e-10) / weights.size * 100
    
    plt.text(0.05, 0.95, f"Std Dev: {std_dev:.6f}\nZero weights: {zero_weights:.2f}%",
             transform=plt.gca().transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Compare validation accuracy across model types
plt.figure(figsize=(12, 6))

# Extract final validation accuracies
val_accuracies = [elastic_net_histories[name].history['val_accuracy'][-1] 
                  for name in elastic_net_models.keys()]
names = list(elastic_net_models.keys())

# Create bar chart
plt.bar(names, val_accuracies, color='lightcoral')
plt.title('Test Accuracy with Different Regularization Approaches')
plt.ylabel('Accuracy')
plt.ylim(0.7, 1.0)
plt.xticks(rotation=45)
plt.grid(axis='y')

for i, v in enumerate(val_accuracies):
    plt.text(i, v + 0.01, f"{v:.4f}", ha='center')

plt.tight_layout()
plt.show()

# Compare validation loss curves
plt.figure(figsize=(12, 6))

for name, history in elastic_net_histories.items():
    plt.plot(history.history['val_loss'], label=name)

plt.title('Validation Loss with Different Regularization Approaches')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 6. Dropout Regularization

Dropout is a regularization technique specific to neural networks. Unlike L1 and L2 which penalize weights, dropout works by randomly "dropping out" (setting to zero) a percentage of neurons during each training iteration.

### How Dropout Works:

1. During training, randomly set a fraction (e.g., 20% or 50%) of neurons to zero
2. Scale the remaining neurons to maintain the expected activation sum
3. During testing/inference, use all neurons but scale their outputs by the dropout probability

### Key Properties of Dropout:

1. **Prevents Co-adaptation**: Neurons cannot rely on specific other neurons being present
2. **Ensemble Effect**: Conceptually similar to training many different networks and averaging them
3. **No specific penalty term**: Doesn't modify the loss function directly

Let's implement dropout and see how it affects our models.

In [ ]:
# Create models with different dropout rates
def create_dropout_models():
    models = {}
    dropout_rates = [0, 0.2, 0.5, 0.8]
    
    for rate in dropout_rates:
        name = f"Dropout ({rate})"
        model = Sequential([
            Dense(128, activation='relu', input_shape=(20,)),
            Dropout(rate),  # First dropout layer after first hidden layer
            Dense(64, activation='relu'),
            Dropout(rate),  # Second dropout layer after second hidden layer
            Dense(1, activation='sigmoid')
        ])
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        models[name] = model
    
    return models

# Train dropout models
dropout_models = create_dropout_models()
dropout_histories = {}

for name, model in dropout_models.items():
    print(f"Training {name}...")
    history = model.fit(
        X_train_scaled, y_train,
        epochs=100,  # Train longer with dropout
        batch_size=32,
        validation_data=(X_test_scaled, y_test),
        verbose=0
    )
    dropout_histories[name] = history

In [ ]:
# Visualize training and validation loss curves for dropout models
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
for name, history in dropout_histories.items():
    plt.plot(history.history['loss'], label=f"{name} - Train")
plt.title('Training Loss with Different Dropout Rates')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for name, history in dropout_histories.items():
    plt.plot(history.history['val_loss'], label=f"{name} - Val")
plt.title('Validation Loss with Different Dropout Rates')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize final training and validation accuracy for dropout models
train_acc = [dropout_histories[name].history['accuracy'][-1] for name in dropout_models.keys()]
val_acc = [dropout_histories[name].history['val_accuracy'][-1] for name in dropout_models.keys()]
names = list(dropout_models.keys())

plt.figure(figsize=(12, 6))

x = np.arange(len(names))
width = 0.35

plt.bar(x - width/2, train_acc, width, label='Training Accuracy', color='skyblue')
plt.bar(x + width/2, val_acc, width, label='Validation Accuracy', color='lightcoral')

plt.title('Training vs Validation Accuracy with Different Dropout Rates')
plt.xlabel('Dropout Rate')
plt.ylabel('Accuracy')
plt.ylim(0.7, 1.0)
plt.xticks(x, names)
plt.legend()
plt.grid(axis='y')

# Add accuracy values on bars
for i, v in enumerate(train_acc):
    plt.text(i - width/2, v + 0.01, f"{v:.3f}", ha='center', fontsize=9)
    
for i, v in enumerate(val_acc):
    plt.text(i + width/2, v + 0.01, f"{v:.3f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

### Visualizing Dropout in Action

To better understand how dropout works, let's create a visual representation of the "dropping out" process on a simple neural network.

In [ ]:
# Create a visual representation of dropout
def visualize_dropout(dropout_rate=0.5):
    plt.figure(figsize=(15, 6))
    
    # Network architecture parameters
    input_nodes = 5
    hidden_nodes = 10
    output_nodes = 2
    
    # Coordinates for nodes
    input_x = np.ones(input_nodes) * 1
    input_y = np.linspace(1, 5, input_nodes)
    
    hidden_x = np.ones(hidden_nodes) * 3
    hidden_y = np.linspace(0.5, 5.5, hidden_nodes)
    
    output_x = np.ones(output_nodes) * 5
    output_y = np.linspace(2, 4, output_nodes)
    
    # Create two subplots: normal network and network with dropout
    for i, title in enumerate(["Regular Network", f"Network with Dropout ({dropout_rate})"]):
        plt.subplot(1, 2, i+1)
        
        # Draw edges between input and hidden layer
        for j in range(input_nodes):
            for k in range(hidden_nodes):
                # For dropout network, randomly skip connections
                if i == 1 and np.random.random() < dropout_rate:
                    continue
                plt.plot([input_x[j], hidden_x[k]], [input_y[j], hidden_y[k]], 'gray', alpha=0.3)
        
        # Draw edges between hidden and output layer
        for j in range(hidden_nodes):
            # For dropout plot, randomly drop hidden nodes
            is_dropped = False
            if i == 1 and np.random.random() < dropout_rate:
                is_dropped = True
            
            for k in range(output_nodes):
                if not is_dropped:
                    plt.plot([hidden_x[j], output_x[k]], [hidden_y[j], output_y[k]], 'gray', alpha=0.3)
        
        # Draw input nodes
        plt.scatter(input_x, input_y, s=100, c='blue', label='Input Layer')
        
        # Draw hidden nodes
        if i == 0:
            # Regular network - all nodes active
            plt.scatter(hidden_x, hidden_y, s=100, c='green', label='Hidden Layer')
        else:
            # Dropout network - some nodes inactive
            active_mask = np.random.random(hidden_nodes) >= dropout_rate
            plt.scatter(hidden_x[active_mask], hidden_y[active_mask], s=100, c='green', label='Active Hidden Neurons')
            plt.scatter(hidden_x[~active_mask], hidden_y[~active_mask], s=100, c='lightgray', label='Dropped Out')
        
        # Draw output nodes
        plt.scatter(output_x, output_y, s=100, c='red', label='Output Layer')
        
        plt.title(title)
        plt.axis('off')
        plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)
    
    plt.tight_layout()
    plt.show()

# Visualize a network with 50% dropout
visualize_dropout(dropout_rate=0.5)

## 7. Implementing L1/L2 Regularization in TensorFlow/Keras

Now that we understand how these regularization techniques work conceptually, let's look at different ways to implement them in TensorFlow/Keras.

### Ways to Apply L1/L2 Regularization:

1. Through layer arguments during construction
2. Using the regularizers API
3. Through activity regularization
4. Custom regularizers

In [ ]:
# Different ways to implement L1/L2 regularization in Keras/TensorFlow

# 1. Layer argument method
model_layer_arg = Sequential([
    Dense(64, activation='relu', input_shape=(20,),
          kernel_regularizer=regularizers.l2(0.01)),  # L2 on weights
    Dense(32, activation='relu',
          kernel_regularizer=regularizers.l2(0.01),   # L2 on weights
          bias_regularizer=regularizers.l2(0.01)),    # L2 on biases
    Dense(1, activation='sigmoid')
])

# 2. Using regularizers API
l1_reg = regularizers.l1(0.01)  # L1 regularizer
l2_reg = regularizers.l2(0.01)  # L2 regularizer
l1_l2_reg = regularizers.l1_l2(l1=0.01, l2=0.01)  # Combined L1+L2

model_reg_api = Sequential([
    Dense(64, activation='relu', input_shape=(20,),
          kernel_regularizer=l1_l2_reg),  
    Dense(32, activation='relu',
          kernel_regularizer=l1_l2_reg),  
    Dense(1, activation='sigmoid')
])

# 3. Activity regularization (regularizes the output of the layer)
model_activity_reg = Sequential([
    Dense(64, activation='relu', input_shape=(20,)),
    layers.ActivityRegularization(l1=0.01),  # L1 on activations
    Dense(32, activation='relu'),
    layers.ActivityRegularization(l2=0.01),  # L2 on activations
    Dense(1, activation='sigmoid')
])

# Print model summaries
print("Model with Layer Argument Regularization:")
model_layer_arg.summary()
print("\nModel with Regularizers API:")
model_reg_api.summary()
print("\nModel with Activity Regularization:")
model_activity_reg.summary()

In [ ]:
# Create custom regularizer that combines L1/L2 with a different weighting scheme
def custom_regularizer(weight_matrix):
    return 0.01 * tf.reduce_sum(tf.abs(weight_matrix)) + \
           0.01 * tf.reduce_sum(tf.square(weight_matrix)) * (1.0 / (1.0 + tf.reduce_sum(tf.abs(weight_matrix))))

# Model with custom regularizer
model_custom_reg = Sequential([
    Dense(64, activation='relu', input_shape=(20,),
          kernel_regularizer=custom_regularizer),
    Dense(32, activation='relu',
          kernel_regularizer=custom_regularizer),
    Dense(1, activation='sigmoid')
])

# Compile the model
model_custom_reg.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model with Custom Regularization:")
model_custom_reg.summary()

## 8. Implementing Dropout in TensorFlow/Keras

Let's examine different ways to use dropout in neural networks and best practices for implementation.

In [ ]:
# Different approaches to implementing dropout

# Standard dropout in a sequential model
model_standard_dropout = Sequential([
    Dense(64, activation='relu', input_shape=(20,)),
    Dropout(0.5),  # 50% dropout after first layer
    Dense(32, activation='relu'),
    Dropout(0.3),  # 30% dropout after second layer
    Dense(1, activation='sigmoid')
])

# Dropout with the Functional API
inputs = Input(shape=(20,))
x = Dense(64, activation='relu')(inputs)
x = Dropout(0.5)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)
model_functional_dropout = Model(inputs=inputs, outputs=outputs)

# Spatial Dropout (often used with CNNs)
# This is for demonstration - would typically use with CNN architecture
model_spatial_dropout = Sequential([
    layers.Reshape((5, 4, 1), input_shape=(20,)),  # Reshape to 5x4 spatial dimensions
    Conv2D(16, kernel_size=(3, 3), activation='relu'),
    layers.SpatialDropout2D(0.5),  # Spatial dropout drops entire feature maps
    Flatten(),
    Dense(1, activation='sigmoid')
])

# Compile the models
for model in [model_standard_dropout, model_functional_dropout, model_spatial_dropout]:
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

print("Standard Dropout Model:")
model_standard_dropout.summary()

print("\nFunctional API Dropout Model:")
model_functional_dropout.summary()

print("\nSpatial Dropout Model (for CNNs):")
model_spatial_dropout.summary()

In [ ]:
# Let's demonstrate the importance of using dropout only during training

# Create a simple model with dropout
dropout_model = Sequential([
    Dense(64, activation='relu', input_shape=(20,)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
dropout_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = dropout_model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    verbose=0
)

# Now let's demonstrate the difference between training and inference mode
dropout_enabled = dropout_model.predict(X_test_scaled[:10], verbose=0)  # Training mode (dropout applied)

# Create predictions using the model with dropout disabled (inference mode)
dropout_model.trainable = False  # This ensures dropout is not applied
dropout_disabled = dropout_model.predict(X_test_scaled[:10], verbose=0)

# Output the results
print("Predictions with dropout (should vary between runs):")
print(dropout_enabled.flatten()[:5])

print("\nPredictions without dropout (inference mode - should be stable):")
print(dropout_disabled.flatten()[:5])

# Note: In Keras, dropout is automatically disabled during inference (model.predict and model.evaluate)
# The example above is to demonstrate the concept rather than how it's actually implemented in the framework

## 9. Comparing Regularization Techniques

Now, let's create a comprehensive comparison of all the regularization techniques we've covered to understand their relative strengths and weaknesses in different contexts.

In [ ]:
# Create a new dataset for comprehensive comparison
# We'll use the California housing dataset
housing = fetch_california_housing()
X = housing.data
y = housing.target

# Normalize target for easier training
y = (y - y.mean()) / y.std()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create models with different regularization techniques
def create_comparative_models():
    models = {}
    
    # Base model with no regularization
    models["No Regularization"] = Sequential([
        Dense(128, activation='relu', input_shape=(X.shape[1],)),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    
    # L1 regularization
    models["L1 Regularization"] = Sequential([
        Dense(128, activation='relu', input_shape=(X.shape[1],),
              kernel_regularizer=regularizers.l1(0.001)),
        Dense(64, activation='relu', kernel_regularizer=regularizers.l1(0.001)),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l1(0.001)),
        Dense(1)
    ])
    
    # L2 regularization
    models["L2 Regularization"] = Sequential([
        Dense(128, activation='relu', input_shape=(X.shape[1],),
              kernel_regularizer=regularizers.l2(0.001)),
        Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        Dense(1)
    ])
    
    # Elastic Net
    models["Elastic Net (L1+L2)"] = Sequential([
        Dense(128, activation='relu', input_shape=(X.shape[1],),
              kernel_regularizer=regularizers.l1_l2(l1=0.0005, l2=0.0005)),
        Dense(64, activation='relu', kernel_regularizer=regularizers.l1_l2(l1=0.0005, l2=0.0005)),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l1_l2(l1=0.0005, l2=0.0005)),
        Dense(1)
    ])
    
    # Dropout
    models["Dropout (0.3)"] = Sequential([
        Dense(128, activation='relu', input_shape=(X.shape[1],)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1)
    ])
    
    # Dropout + L2 (combined approach)
    models["Dropout + L2"] = Sequential([
        Dense(128, activation='relu', input_shape=(X.shape[1],),
              kernel_regularizer=regularizers.l2(0.001)),
        Dropout(0.3),
        Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        Dropout(0.3),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        Dropout(0.3),
        Dense(1)
    ])
    
    # Compile all models
    for model in models.values():
        model.compile(
            optimizer='adam',
            loss='mse',
            metrics=['mae']
        )
    
    return models

# Create models for comparison
comp_models = create_comparative_models()
comp_histories = {}

# Train each model
for name, model in comp_models.items():
    print(f"Training {name}...")
    history = model.fit(
        X_train_scaled, y_train,
        epochs=50,
        batch_size=32,
        validation_data=(X_test_scaled, y_test),
        verbose=0
    )
    comp_histories[name] = history
    
    # Evaluate on test set
    test_loss, test_mae = model.evaluate(X_test_scaled, y_test, verbose=0)
    print(f"{name} - Test MAE: {test_mae:.4f}")

In [ ]:
# Visualize validation loss over epochs
plt.figure(figsize=(12, 6))

for name, history in comp_histories.items():
    plt.plot(history.history['val_loss'], label=name)

plt.title('Validation Loss by Regularization Type')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.grid(True)
plt.show()

# Compare final validation MAE
val_mae = [comp_histories[name].history['val_mae'][-1] for name in comp_models.keys()]
names = list(comp_models.keys())

plt.figure(figsize=(12, 6))
plt.bar(names, val_mae, color='lightblue')
plt.title('Validation Mean Absolute Error by Regularization Type')
plt.ylabel('Mean Absolute Error')
plt.xticks(rotation=45)
plt.grid(axis='y')

for i, v in enumerate(val_mae):
    plt.text(i, v + 0.01, f"{v:.4f}", ha='center')

plt.tight_layout()
plt.show()

## 10. Visualizing the Effects of Regularization

To get a deeper understanding of how regularization affects our models, let's create some visualizations that show the impact on weights, decision boundaries, and learned features.

In [ ]:
# Extract weights from the first layer of each model for comparison
def extract_first_layer_weights(models):
    weights = {}
    for name, model in models.items():
        weights[name] = model.layers[0].get_weights()[0].flatten()
    return weights

# Get weights from our comparative models
model_weights = extract_first_layer_weights(comp_models)

# Visualize weight distributions
plt.figure(figsize=(15, 10))

for i, (name, weights) in enumerate(model_weights.items()):
    plt.subplot(3, 2, i+1)
    
    # Plot histogram of weights
    plt.hist(weights, bins=50, alpha=0.7)
    plt.title(f"{name}\nWeight Distribution")
    plt.xlabel("Weight Value")
    plt.ylabel("Count")
    
    # Calculate and display statistics
    mean = np.mean(weights)
    std = np.std(weights)
    zero_count = np.sum(np.abs(weights) < 1e-10)
    zero_percent = zero_count / len(weights) * 100
    
    stats = f"Mean: {mean:.6f}\nStd Dev: {std:.6f}\nZero Weights: {zero_percent:.2f}%"
    plt.text(0.05, 0.95, stats, transform=plt.gca().transAxes, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Create a heatmap to visualize weight importance across features
# We'll use the absolute values of weights to represent importance

# Get feature names
feature_names = housing.feature_names

# Create a DataFrame to store weight magnitudes
weights_df = pd.DataFrame()

for name, model in comp_models.items():
    # Get weights from the first layer
    weights = model.layers[0].get_weights()[0]  # Shape: (n_features, n_neurons)
    
    # Calculate the average absolute weight for each feature
    avg_abs_weights = np.mean(np.abs(weights), axis=1)
    weights_df[name] = avg_abs_weights

# Add feature names
weights_df.index = feature_names

# Plot heatmap
plt.figure(figsize=(14, 8))
sns.heatmap(weights_df, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("Average Absolute Weight Magnitude by Feature and Regularization Type")
plt.ylabel("Feature")
plt.xlabel("Regularization Method")
plt.tight_layout()
plt.show()

In [ ]:
# Create a feature importance visualization
plt.figure(figsize=(14, 8))

# Normalize weights for better comparison
normalized_weights = weights_df.copy()
for col in normalized_weights.columns:
    max_val = normalized_weights[col].max()
    if max_val > 0:  # Avoid division by zero
        normalized_weights[col] = normalized_weights[col] / max_val

# Plot normalized feature importance
ax = normalized_weights.plot(kind='bar', figsize=(14, 8))
plt.title("Normalized Feature Importance by Regularization Type")
plt.ylabel("Relative Importance")
plt.xlabel("Feature")
plt.legend(title="Regularization Method")
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## 11. Practical Tips for Using Regularization

Based on our experiments and analysis, here are some practical guidelines for using regularization techniques effectively in your deep learning models.

### When to Use Each Regularization Technique

1. **L1 Regularization (Lasso)**
   - Use when you suspect many input features are irrelevant
   - When feature selection is desirable (sparse models)
   - When interpretability is important
   - Typical values: 0.0001 to 0.01

2. **L2 Regularization (Ridge)**
   - General-purpose regularization for most neural networks
   - When dealing with correlated features
   - When you want to prevent large weights without forcing them to zero
   - Typical values: 0.0001 to 0.01

3. **Elastic Net (L1+L2)**
   - When you want benefits of both L1 and L2
   - In high-dimensional spaces with correlations between features
   - When both feature selection and weight decay are desirable
   - Typical values: 0.0001 to 0.005 for each component

4. **Dropout**
   - Primarily for neural networks with many parameters
   - When ensemble-like behavior is desired
   - To prevent co-adaptation of features
   - Typical values:
     - 0.2 to 0.3 for input layers
     - 0.5 for hidden layers
     - Generally not used on output layer

### General Guidelines for Regularization

- **Start with a baseline model** without regularization to understand your problem
- **Use cross-validation** to tune regularization hyperparameters
- **Layer-specific regularization** can be more effective than applying the same strength to all layers
- **Combine techniques** for complex models (e.g., L2 with dropout often works well)
- **Monitor validation metrics** to ensure regularization is helping, not hurting
- **Stronger regularization** for smaller datasets or complex models
- **Weaker regularization** for larger datasets or simpler models

### Common Mistakes to Avoid

1. **Applying too much regularization** - Can lead to underfitting
2. **Not scaling input data** - Regularization works better with standardized inputs
3. **Applying dropout incorrectly** - Remember it's only active during training
4. **Forgetting that regularization adds losses** - Total loss will be higher with regularization even when model is better
5. **Using the same regularization strategy for all problems** - Different tasks may need different approaches

In [ ]:
# Let's implement a simple function to help choose regularization
def suggest_regularization(n_samples, n_features, complexity='medium', correlation='medium'):
    """
    Suggests regularization techniques based on dataset characteristics.
    
    Parameters:
    -----------
    n_samples: int
        Number of training examples
    n_features: int
        Number of input features
    complexity: str
        Model complexity: 'low', 'medium', or 'high'
    correlation: str
        Expected correlation between features: 'low', 'medium', or 'high'
    
    Returns:
    --------
    dict: Suggested regularization techniques and values
    """
    suggestions = {}
    
    # Calculate sample to feature ratio
    ratio = n_samples / n_features
    
    # Base suggestions on ratio and other parameters
    if ratio < 5:  # Very few samples compared to features
        suggestions['primary'] = "L1 regularization"
        suggestions['l1_strength'] = 0.01
        suggestions['dropout_rate'] = 0.5
        suggestions['notes'] = "High risk of overfitting. Use strong regularization."
    
    elif ratio < 20:  # Moderate sample to feature ratio
        if correlation == 'high':
            suggestions['primary'] = "Elastic Net"
            suggestions['l1_strength'] = 0.001
            suggestions['l2_strength'] = 0.01
            suggestions['dropout_rate'] = 0.3
            suggestions['notes'] = "Use L2-dominant elastic net for correlated features."
        else:
            suggestions['primary'] = "L1 + Dropout"
            suggestions['l1_strength'] = 0.001
            suggestions['dropout_rate'] = 0.3
            suggestions['notes'] = "Use L1 for feature selection and dropout for regularization."
    
    else:  # Many samples compared to features
        if complexity == 'high':
            suggestions['primary'] = "Dropout + L2 (light)"
            suggestions['l2_strength'] = 0.0005
            suggestions['dropout_rate'] = 0.2
            suggestions['notes'] = "Focus on preventing co-adaptation with dropout."
        else:
            suggestions['primary'] = "L2 regularization (light)"
            suggestions['l2_strength'] = 0.0001
            suggestions['notes'] = "Light regularization should be sufficient."
    
    return suggestions

# Example usage
datasets = [
    {"name": "Small dataset, many features", "samples": 100, "features": 50, 
     "complexity": "medium", "correlation": "medium"},
    {"name": "Standard tabular dataset", "samples": 1000, "features": 20, 
     "complexity": "medium", "correlation": "low"},
    {"name": "Image classification", "samples": 10000, "features": 1000, 
     "complexity": "high", "correlation": "high"},
    {"name": "Large text dataset", "samples": 100000, "features": 10000, 
     "complexity": "high", "correlation": "high"}
]

# Print suggestions for each example dataset
for dataset in datasets:
    suggestions = suggest_regularization(
        dataset["samples"], 
        dataset["features"],
        dataset["complexity"],
        dataset["correlation"]
    )
    
    print(f"Dataset: {dataset['name']}")
    print(f"  - {dataset['samples']} samples, {dataset['features']} features")
    print(f"  - Complexity: {dataset['complexity']}, Feature correlation: {dataset['correlation']}")
    print(f"  - Primary recommendation: {suggestions['primary']}")
    for key, value in suggestions.items():
        if key not in ['primary', 'notes']:
            print(f"  - {key}: {value}")
    print(f"  - Note: {suggestions['notes']}")
    print()

## Summary and Key Takeaways

In this notebook, we've explored various regularization techniques for deep learning models:

1. **L1 Regularization (Lasso)**
   - Adds the sum of absolute weights to the loss function
   - Promotes sparsity and performs feature selection
   - Drives unimportant weights to exactly zero

2. **L2 Regularization (Ridge)**
   - Adds the sum of squared weights to the loss function
   - Prevents large weights but keeps most features
   - Works well with correlated features

3. **Elastic Net**
   - Combines L1 and L2 regularization
   - Gets benefits of both approaches
   - Allows fine-tuning the balance between sparsity and weight decay

4. **Dropout**
   - Randomly deactivates neurons during training
   - Prevents co-adaptation and creates an ensemble effect
   - Simple but effective technique specific to neural networks

**Key Insights**:

- Regularization is essential for preventing overfitting in complex models
- The choice of regularization depends on dataset size, feature relationships, and model complexity
- Combining regularization techniques can be powerful (e.g., dropout with L2)
- Proper hyperparameter tuning is crucial for effective regularization
- Always monitor validation metrics to ensure regularization is helping

By applying these regularization techniques appropriately, you can build deep learning models that generalize better to unseen data and are more robust in production environments.